In [1]:
# import libraries
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# read in main data includes geocoded coordinates
stops = pd.read_csv("../data/raw/policing_imputed_coords.csv.gz", compression='gzip')
# drop column
stops = stops.drop(columns=['Unnamed: 0'])

/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel_33583/4074306894.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  stops = pd.read_csv("../data/raw/policing_imputed_coords.csv.gz", compression='gzip')


In [3]:
#fix time
stops["time"] = pd.to_timedelta(stops["time"], unit="s")
stops["time"] = (pd.Timestamp("today").normalize() + stops["time"]).dt.strftime("%H:%M:%S")

In [4]:
# ensure date is in the correct format
stops['date'] = pd.to_datetime(stops['date'])

In [5]:
# check the number of observations without date to make sure is not excessive
stops["date"].isna().sum()
# drop these rows
stops.dropna(subset=['date'], inplace=True)

In [6]:
# drop rows without lat/long since this is neccessary for our merge
stops.dropna(subset=['final_lat','final_lng'], inplace=True)

In [7]:
# check 5 year span with greatest amount of observations remaining
df_10 = stops[(stops['date'].dt.year >= 2010) & (stops['date'].dt.year <= 2014)]
df_11 = stops[(stops['date'].dt.year >= 2011) & (stops['date'].dt.year <= 2015)]
df_12 = stops[(stops['date'].dt.year >= 2012) & (stops['date'].dt.year <= 2016)]
df_13 = stops[(stops['date'].dt.year >= 2013) & (stops['date'].dt.year <= 2017)]
print(len(df_10), len(df_11), len(df_12), len(df_13))

210137 277310 270134 261770


In [8]:
df_11

,raw_row_number,date,time,location,lat,lng,district,zone,subject_age,subject_race,...,search_basis,reason_for_stop,vehicle_color,vehicle_make,vehicle_model,vehicle_year,raw_actions_taken,raw_subject_race,final_lat,final_lng
378,149223,2012-01-01,13:01:00,Barracks St & N Peters St,29.960840,-90.057711,8,E,35.0,black,...,NaN,TRAFFIC VIOLATION,GRAY,FORD,NaN,2003.0,Stop Results: Citation Issued;Subject Type: Dr...,BLACK,29.960840,-90.057711
379,149149,2012-01-01,13:08:00,Dumaine St & N Rampart St,29.962113,-90.066910,8,E,26.0,white,...,NaN,TRAFFIC VIOLATION,WHITE,BUICK,RIVIERA,1998.0,Stop Results: Citation Issued;Subject Type: Dr...,WHITE,29.962113,-90.066910
380,149086,2012-01-01,01:11:00,S Carrollton Ave & Olive St,29.963036,-90.112752,2,T,27.0,black,...,other,TRAFFIC VIOLATION,TAN,BUICK,LESABRE,2000.0,Stop Results: Verbal Warning;Search Occurred: ...,BLACK,29.963036,-90.112752
381,149225,2012-01-01,13:12:00,Barracks St & N Peters St,29.960840,-90.057711,8,E,19.0,white,...,NaN,TRAFFIC VIOLATION,SILVER,TOYOTA,CAMRY,2000.0,Stop Results: Citation Issued;Subject Type: Dr...,WHITE,29.960840,-90.057711
382,149081,2012-01-01,01:13:00,S Rendon & Washington,29.961357,-90.099890,2,V,67.0,white,...,NaN,TRAFFIC VIOLATION,NaN,TOYOTA,CAMRY,2011.0,Stop Results: Verbal Warning;Subject Type: Dri...,WHITE,29.961357,-90.099890
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511804,377179,2015-12-31,00:32:00,Calhoun St & S Claiborne Ave,29.945885,-90.113974,2,M,55.0,white,...,consent,SUSPECT PERSON,NaN,NaN,NaN,NaN,Stop Results: No action taken;Subject Type: Pe...,WHITE,29.945885,-90.113974
511805,377463,2015-12-31,12:37:00,Read Blvd & Lake Forest Blvd,30.032609,-89.972994,7,I,21.0,black,...,other,CALL FOR SERVICE,BLACK,JEEP,LIBERTY,2004.0,Stop Results: Citation Issued;Stop Results: Ph...,BLACK,30.032609,-89.972994
511806,377249,2015-12-31,00:45:00,007XX Conti St,NaN,NaN,8,C,65.0,black,...,NaN,CRIMINAL VIOLATION,NaN,NaN,NaN,NaN,Stop Results: Physical Arrest;Subject Type: Pe...,BLACK,29.955666,-90.066982
511807,377218,2015-12-31,12:47:00,Lafreniere St & London Ave,29.993291,-90.068609,1,L,21.0,black,...,other,CALL FOR SERVICE,NaN,NaN,NaN,NaN,Stop Results: Physical Arrest;Subject Type: Pe...,BLACK,29.993291,-90.068609


In [9]:
# check missing data for main data
missing = df_11.isna().sum()
missing_percent = (missing / len(df_11)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_percent': missing_percent})
missing_df_sorted = missing_df.sort_values(by="missing_percent", ascending=False)
missing_df_sorted

,missing_count,missing_percent
search_basis,230825,83.237171
contraband_found,230825,83.237171
contraband_weapons,230825,83.237171
contraband_drugs,230825,83.237171
vehicle_model,140142,50.536223
vehicle_year,133527,48.150806
vehicle_color,132383,47.738271
vehicle_make,130934,47.215751
lat,101678,36.665825
lng,101678,36.665825


In [10]:
# read acs data
dem = pd.read_csv("../data/raw/acs/2015DEM.csv")
edu = pd.read_csv("../data/raw/acs/2015EDU.csv")
inc = pd.read_csv("../data/raw/acs/2015INC.csv")

In [11]:
# transpose demographic data
dem = dem.T
# fix column names for all data
dem.columns = dem.iloc[0]
dem = dem[1:]
edu.columns = edu.iloc[0]
edu = edu[1:]
inc.columns = inc.iloc[0]
inc = inc[1:]

In [12]:
# only get census tract data estimate, exclude margin of error
dem = dem[dem.index.str.strip().str.endswith("Estimate")]
# correct index
dem = dem.reset_index()
dem = dem.rename(columns={"index": "Geographic Area Name"})

In [13]:
# choose columns to keep
dem.columns
cols = ['Geographic Area Name', '\xa0\xa0\xa0\xa0Total population', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0White',
 '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Black or African American',
 '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0American Indian and Alaska Native','\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Asian', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Native Hawaiian and Other Pacific Islander', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Some other race', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Male',
 '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Female']
dem_sub = dem[cols]

In [14]:
# remove commas and spaces then convert to numeric
dem_sub = dem_sub.loc[:, ~dem_sub.columns.duplicated()].copy()
cols_to_convert = dem_sub.columns.drop("Geographic Area Name")
for col in cols_to_convert:
    dem_sub[col] = pd.to_numeric(dem_sub[col].astype(str).str.replace(",", "").str.strip(), errors='coerce')

In [15]:
# check missing data for demographic data
missing = dem_sub.isna().sum()
missing_percent = (missing / len(dem_sub)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_percent': missing_percent})
missing_df_sorted = missing_df.sort_values(by="missing_percent", ascending=False)
missing_df_sorted

,missing_count,missing_percent
Label (Grouping),,
Geographic Area Name,0,0.0
Total population,0,0.0
White,0,0.0
Black or African American,0,0.0
American Indian and Alaska Native,0,0.0
Asian,0,0.0
Native Hawaiian and Other Pacific Islander,0,0.0
Some other race,0,0.0
Male,0,0.0


In [16]:
# choose columns to keep
edu.columns.to_list()
cols_to_keep = [
    'Geographic Area Name',
    'Total!!Estimate!!Population 25 years and over',
    'Total!!Estimate!!Population 25 years and over!!Less than 9th grade',
    'Total!!Estimate!!Population 25 years and over!!9th to 12th grade, no diploma',
    'Total!!Estimate!!Population 25 years and over!!High school graduate (includes equivalency)',
    'Total!!Estimate!!Population 25 years and over!!Some college, no degree',
    "Total!!Estimate!!Population 25 years and over!!Associate's degree",
    "Total!!Estimate!!Population 25 years and over!!Bachelor's degree",
    'Total!!Estimate!!Population 25 years and over!!Graduate or professional degree'
]
edu_subset = edu[cols_to_keep]

In [17]:
# skip geographic name
for col in cols_to_keep[1:]:  # skip geographic name
    edu_subset[col] = pd.to_numeric(edu_subset[col].astype(str).str.replace(',', ''), errors='coerce')

/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel_33583/3777716917.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edu_subset[col] = pd.to_numeric(edu_subset[col].astype(str).str.replace(',', ''), errors='coerce')
/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel_33583/3777716917.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edu_subset[col] = pd.to_numeric(edu_subset[col].astype(str).str.replace(',', ''), errors='coerce')
/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel

In [18]:
# check missing data for education data
missing = edu_subset.isna().sum()
missing_percent = (missing / len(edu_subset)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_percent': missing_percent})
missing_df_sorted = missing_df.sort_values(by="missing_percent", ascending=False)
missing_df_sorted

,missing_count,missing_percent
0,,
Geographic Area Name,0,0.0
Total!!Estimate!!Population 25 years and over,0,0.0
Total!!Estimate!!Population 25 years and over!!Less than 9th grade,0,0.0
"Total!!Estimate!!Population 25 years and over!!9th to 12th grade, no diploma",0,0.0
Total!!Estimate!!Population 25 years and over!!High school graduate (includes equivalency),0,0.0
"Total!!Estimate!!Population 25 years and over!!Some college, no degree",0,0.0
Total!!Estimate!!Population 25 years and over!!Associate's degree,0,0.0
Total!!Estimate!!Population 25 years and over!!Bachelor's degree,0,0.0
Total!!Estimate!!Population 25 years and over!!Graduate or professional degree,0,0.0


In [19]:
# choose columns to keep
inc.columns.to_list()
cols_to_keep = [
    'Geography','Geographic Area Name', 'Households!!Estimate!!Total','Households!!Estimate!!Median income (dollars)','Households!!Estimate!!Mean income (dollars)'
]
inc_subset = inc[cols_to_keep]

In [20]:
# skip geographic name
for col in cols_to_keep[2:]:  # skip geographic name
    inc_subset[col] = pd.to_numeric(inc_subset[col].astype(str).str.replace(',', ''), errors='coerce')

/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel_33583/85663157.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inc_subset[col] = pd.to_numeric(inc_subset[col].astype(str).str.replace(',', ''), errors='coerce')
/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel_33583/85663157.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inc_subset[col] = pd.to_numeric(inc_subset[col].astype(str).str.replace(',', ''), errors='coerce')
/var/folders/b7/4yf7b9tn1t1ck0mv8h93dssc0000gn/T/ipykernel_335

In [21]:
# check missing data for income data
missing = inc_subset.isna().sum()
missing_percent = (missing / len(inc_subset)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_percent': missing_percent})
missing_df_sorted = missing_df.sort_values(by="missing_percent", ascending=False)
missing_df_sorted

,missing_count,missing_percent
0,,
Households!!Estimate!!Mean income (dollars),6,3.389831
Households!!Estimate!!Median income (dollars),6,3.389831
Geography,0,0.000000
Households!!Estimate!!Total,0,0.000000
Geographic Area Name,0,0.000000


In [22]:
# clean up text
dem_sub['Geographic Area Name'] = (
    dem_sub['Geographic Area Name']
        .str.replace('!!Estimate', '', regex=False)
        .str.strip()
)

In [23]:
# combine data
# merge edu and income first
edu_inc = edu_subset.merge(inc_subset, on='Geographic Area Name', how='left')
# merge with demographics
full_census = edu_inc.merge(dem_sub, on='Geographic Area Name', how='left')

In [24]:
# remove census geographic prefix before "US"
full_census['Geography'] = full_census['Geography'].str.split('US').str[-1]

In [25]:
# read in shapefile data
tracts = gpd.read_file("../data/raw/shapefiles/tl_2015_22_tract.shp")

In [26]:
# filter tracts that are in census data
tracts_filtered = tracts[tracts['GEOID'].isin(full_census['Geography'])]

In [27]:
# spatial data and inner join
tracts_filtered = tracts_filtered.to_crs(epsg=4326)
geometry = [Point(xy) for xy in zip(df_11.final_lng, df_11.final_lat)]
points = gpd.GeoDataFrame(df_11, geometry=geometry, crs="EPSG:4326")
spatial_join = gpd.sjoin(points, tracts_filtered, how="inner", predicate="within")

In [28]:
# make names match for merge
full_census = full_census.rename(columns={'Geography': 'GEOID'})
# merge stop data
stops = spatial_join.merge(full_census, on='GEOID', how='left')

In [29]:
# create outcome variable
stops['outcome'] = 'None'  # default
stops.loc[stops['citation_issued'] == True, 'outcome'] = 'Citation'
stops.loc[stops['arrest_made'] == True, 'outcome'] = 'Arrest'
stops.loc[stops['warning_issued'] == True, 'outcome'] = 'Warning'

In [30]:
stops.columns

Index(['raw_row_number', 'date', 'time', 'location', 'lat', 'lng', 'district',
       'zone', 'subject_age', 'subject_race', 'subject_sex',
       'officer_assignment', 'type', 'arrest_made', 'citation_issued',
       'warning_issued', 'outcome', 'contraband_found', 'contraband_drugs',
       'contraband_weapons', 'frisk_performed', 'search_conducted',
       'search_person', 'search_vehicle', 'search_basis', 'reason_for_stop',
       'vehicle_color', 'vehicle_make', 'vehicle_model', 'vehicle_year',
       'raw_actions_taken', 'raw_subject_race', 'final_lat', 'final_lng',
       'geometry', 'index_right', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID',
       'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT',
       'INTPTLON', 'Geographic Area Name',
       'Total!!Estimate!!Population 25 years and over',
       'Total!!Estimate!!Population 25 years and over!!Less than 9th grade',
       'Total!!Estimate!!Population 25 years and over!!9th to 12th grade, no diploma',
  

In [31]:
stops.dtypes

raw_row_number                                                object
date                                                  datetime64[ns]
time                                                          object
location                                                      object
lat                                                          float64
                                                           ...      
        Asian                                                float64
        Native Hawaiian and Other Pacific Islander           float64
        Some other race                                      float64
        Male                                                 float64
        Female                                               float64
Length: 69, dtype: object

In [32]:
# create y arrest variable for modeling
stops["y_arrest"] = (stops["outcome"].astype(str).str.lower() == "arrest").astype(int)
# date column
stops["stop_date"] = pd.to_datetime(stops["date"], errors="coerce")
# parse time
def parse_time(t):
    try:
        # float to seconds
        return pd.to_timedelta(float(t), unit="s")
    except:
        try:
            # parse string
            return pd.to_timedelta(t)
        except:
            return pd.NaT
# create time to extract variables
stops["stop_time"] = stops["time"].apply(parse_time)
# create datetime
stops["stop_datetime"] = stops["stop_date"] + stops["stop_time"]
# extract features, dow: monday=0
stops["stop_hour"] = stops["stop_datetime"].dt.hour
stops["stop_dow"] = stops["stop_datetime"].dt.dayofweek
stops["stop_month"] = stops["stop_datetime"].dt.month
stops["stop_year"] = stops["stop_datetime"].dt.year


In [33]:
stops.to_csv("../data/processed/stops_clean.csv.gz", index=False)
